# RVCBench Qwen3-TTS Quickstart

Minimal end-to-end walkthrough for running **RVCBench** with **Qwen3-TTS** (zero-shot voice cloning).

What this notebook does:
1. Locates (or clones) the RVCBench repository — all paths stay **inside** the repo
2. Installs the Qwen3-TTS dependencies
3. Downloads a LibriTTS subset from `Nanboy/RVCBench` on Hugging Face into `{repo}/data/`
4. Runs `run_vc.py` end-to-end — outputs land in `{repo}/results/`
5. Displays the generated audio and metrics

**Requirements:**
- A GPU runtime (CUDA)
- `conda` environment named `qwen3` with `qwen-tts` installed, **or** run the pip install cell below
- A Hugging Face account (Qwen3-TTS weights are gated — accept access at the model page first)

## 1. Locate the repository

If the notebook is already inside the repo (local server), `REPO_DIR` is set to the repo root.  
If running on Colab (or any machine without the repo), it is cloned first.  
**All data, checkpoints, and results will be written inside `REPO_DIR`.**

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Nanboy-Ronan/RVCBench.git"

def _find_repo_root() -> Path:
    """Walk up from the notebook location until we find run_vc.py."""
    candidate = Path(os.getcwd()).resolve()
    for parent in [candidate] + list(candidate.parents):
        if (parent / "run_vc.py").exists():
            return parent
    return None

REPO_DIR = _find_repo_root()

if REPO_DIR is None:
    # Not inside the repo — clone it next to wherever we are
    clone_target = Path(os.getcwd()) / "RVCBench"
    if not clone_target.exists():
        os.system(f"git clone {REPO_URL} {clone_target}")
    REPO_DIR = clone_target

%cd {REPO_DIR}
print("REPO_DIR:", REPO_DIR)
assert (REPO_DIR / "run_vc.py").exists(), "run_vc.py not found — check REPO_DIR"

## 2. Install dependencies

In [ ]:
%%bash
set -euxo pipefail
python -m pip install --upgrade pip
python -m pip install -q \
  hydra-core omegaconf pandas pyarrow \
  jiwer openai-whisper pymcd librosa soundfile \
  huggingface_hub
python -m pip install -q qwen-tts

## 3. Authenticate with Hugging Face

Qwen3-TTS weights are gated. Before running:
1. Accept terms at `https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-Base`
2. Create a HF token with read access.

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token with access to Qwen3-TTS: ")

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF login configured.")

## 4. Download benchmark data into `{repo}/data/`

Data is fetched from `Nanboy/RVCBench` on Hugging Face and written to `{REPO_DIR}/data/Libritts/`.  
We download one speaker to keep the demo small; the benchmark loader can pull the full dataset automatically.

In [ ]:
from huggingface_hub import snapshot_download

HF_CONFIG  = "Libritts"   # HF config name — matches configs/dataset/libritts.yaml
SPEAKER_ID = "1089"        # single LibriTTS speaker for this demo

# All data lands inside the repo under data/
DATA_DIR     = REPO_DIR / "data"
DATASET_ROOT = DATA_DIR / HF_CONFIG

snapshot_download(
    repo_id="Nanboy/RVCBench",
    repo_type="dataset",
    local_dir=str(DATA_DIR),
    allow_patterns=[
        f"{HF_CONFIG}/metadata.parquet",
        f"{HF_CONFIG}/audios/{SPEAKER_ID}/*.wav",
    ],
)

print("Dataset root:", DATASET_ROOT)
wav_files = sorted((DATASET_ROOT / "audios" / SPEAKER_ID).glob("*.wav"))
print(f"Downloaded {len(wav_files)} wav files for speaker {SPEAKER_ID}")
for p in wav_files[:5]:
    print(" -", p.name)

## 5. Configure the run

In [ ]:
import torch

os.environ["DATA_LOADER_WORKERS"] = "0"  # avoid fork issues in notebooks

DEVICE      = "cuda:0" if torch.cuda.is_available() else "cpu"
MAX_SAMPLES = 5  # increase for more thorough benchmarking

print(f"REPO_DIR     = {REPO_DIR}")
print(f"DATASET_ROOT = {DATASET_ROOT}")
print(f"DEVICE       = {DEVICE}")
print(f"MAX_SAMPLES  = {MAX_SAMPLES}")

## 6. Run the benchmark end-to-end

`run_vc.py` is invoked from `REPO_DIR`, so generated audio and metrics are written to
`{REPO_DIR}/results/qwen3_tts_ots_on_libritts/`.  
The first run downloads the Qwen3-TTS checkpoint (≈ 3.5 GB) and caches it under the HF hub cache; subsequent runs start immediately.

In [ ]:
%%bash -s "$DEVICE" "$MAX_SAMPLES" "$DATASET_ROOT" "$SPEAKER_ID"
set -euxo pipefail

DEVICE=$1
MAX_SAMPLES=$2
DATASET_ROOT=$3
SPEAKER_ID=$4

python run_vc.py \
  --config-name ots_vc/clean/libritts/qwen3_tts_ots \
  device=${DEVICE} \
  dataset.root_path=${DATASET_ROOT} \
  dataset.use_hf_dataset=false \
  dataset.speaker_id=${SPEAKER_ID} \
  adversary.max_samples=${MAX_SAMPLES} \
  batch_size=1

## 7. Inspect results

All outputs are inside `{REPO_DIR}/results/`.

In [ ]:
import json
from IPython.display import Audio, display

results_dir  = REPO_DIR / "results" / "qwen3_tts_ots_on_libritts"
latest_run   = max(results_dir.glob("*"), key=lambda p: p.stat().st_mtime)
metrics_path = latest_run / "metrics.json"
generated_dir = latest_run / "generated_audio"

print("Latest result dir:", latest_run)

metrics  = json.loads(metrics_path.read_text())
gen_eval = metrics.get("generation_evaluation", {})
print("\n=== Generation Metrics ===")
print(json.dumps(gen_eval, indent=2)[:3000])

generated_wavs = sorted(generated_dir.rglob("*.wav"))
print(f"\nGenerated {len(generated_wavs)} audio file(s)")

In [ ]:
if generated_wavs:
    print("Playing:", generated_wavs[0].name)
    display(Audio(str(generated_wavs[0])))
else:
    print("No generated audio found.")

## Tips & common overrides

| Goal | Override |
|---|---|
| More samples | `adversary.max_samples=100` |
| Different dataset | swap `libritts` for `aishell`, `vctk`, etc. in `--config-name` |
| Different speaker | `dataset.speaker_id=1272` |
| Full dataset via HF (auto-download) | omit `dataset.use_hf_dataset=false` and `dataset.root_path` overrides |
| Local checkpoint | `adversary.checkpoint_path=./checkpoints/Qwen3-TTS` |
| Flash-attention-2 | `adversary.use_flash_attn2=true` (requires `flash_attn`) |
| Different language | `adversary.language=Chinese` |
| Eval-only | `vc.evaluate_only=true vc.evaluation.generated_audio_dir=...` |

See `configs/ots_vc/clean/libritts/qwen3_tts_ots.yaml` for the full list of adversary parameters.